# Current test macro F1

HEST reports the mean test macro F1 across the five fast samples. PhenoCell reports test macro F1 for its all-class and immune tasks. Distillation v1 is shown for penultimate and probability features at K=256 and K=64. A pending HEST probe is left blank until all five samples finish.

In [ ]:
from pathlib import Path

import pandas as pd


ROOT = Path("/lustre/scratch126/cellgen/lotfollahi/so25/projects/cellbridge")

RUNS = [
    {
        "name": "UNI baseline",
        "hest": ROOT
        / "outputs/distill_v0_cosine_only/label_test_fast_probe_metrics.csv",
        "hest_arm": "uni_baseline",
        "ood": ROOT
        / "outputs/evaluation/phenocell_crop96_distill_v0/probes/summary_all.csv",
        "ood_arm": "uni",
    },
    {
        "name": "Cosine only",
        "hest": ROOT
        / "outputs/distill_v0_cosine_only/label_test_fast_probe_metrics.csv",
        "hest_arm": "distilled",
        "ood": ROOT
        / "outputs/evaluation/phenocell_crop96_cosine_only/probes/summary_distilled.csv",
    },
    {
        "name": "VICReg mild",
        "hest": ROOT
        / "outputs/distill_v0_vicreg_mild/label_test_fast_probe_metrics.csv",
        "hest_arm": "distilled",
        "ood": ROOT
        / "outputs/evaluation/phenocell_crop96_vicreg_mild/probes/summary_distilled.csv",
    },
    {
        "name": "VICReg strong",
        "hest": ROOT
        / "outputs/distill_v0_vicreg_strong/label_test_fast_probe_metrics.csv",
        "hest_arm": "distilled",
        "ood": ROOT
        / "outputs/evaluation/phenocell_crop96_vicreg_strong/probes/summary_distilled.csv",
    },
    {
        "name": "v1, K=256 — penultimate",
        "hest": ROOT / "outputs/distill_v1/label_test_fast_probe_metrics.csv",
        "hest_arm": "prototype_penultimate_512",
        "ood": ROOT
        / "outputs/evaluation/phenocell_crop96_v1_penultimate/probes/summary_distilled.csv",
    },
    {
        "name": "v1, K=256 — probabilities",
        "hest": ROOT / "outputs/distill_v1/label_test_fast_probe_metrics.csv",
        "hest_arm": "prototype_probs_256",
        "ood": ROOT
        / "outputs/evaluation/phenocell_crop96_v1_probs/probes/summary_distilled.csv",
    },
    {
        "name": "v1, K=64 — penultimate",
        "hest": ROOT / "outputs/distill_v1_k64",
        "hest_arm": "prototype_penultimate_512",
        "ood": ROOT
        / "outputs/evaluation/phenocell_crop96_v1_k64_penultimate/probes/summary_distilled.csv",
    },
    {
        "name": "v1, K=64 — probabilities",
        "hest": ROOT / "outputs/distill_v1_k64",
        "hest_arm": "prototype_probs_64",
        "ood": ROOT
        / "outputs/evaluation/phenocell_crop96_v1_k64_probs/probes/summary_distilled.csv",
    },
]


def hest_macro_f1(path, arm):
    if path.is_file():
        table = pd.read_csv(path)
        values = table.loc[table["arm"] == arm, "macro_f1"]
    else:
        metric_paths = sorted(
            path.glob(f"label_test_fast_probes/*/{arm}/terra/metrics.csv")
        )
        values = pd.Series(
            [
                pd.read_csv(metric_path)["macro_f1"].iloc[0]
                for metric_path in metric_paths
            ]
        )
    if len(values) != 5:
        return float("nan")
    return values.mean()


def phenocell_macro_f1(path, task, arm="distilled"):
    table = pd.read_csv(path)
    values = table.loc[(table["task"] == task) & (table["arm"] == arm), "test_macro_f1"]
    if len(values) != 1:
        raise ValueError(
            f"Expected one PhenoCell result for {task}, found {len(values)}"
        )
    return values.iloc[0]


results = pd.DataFrame(
    [
        {
            "Method": run["name"],
            "HEST fast (5-sample mean)": hest_macro_f1(run["hest"], run["hest_arm"]),
            "PhenoCell all": phenocell_macro_f1(
                run["ood"], "all", run.get("ood_arm", "distilled")
            ),
            "PhenoCell immune": phenocell_macro_f1(
                run["ood"], "immune", run.get("ood_arm", "distilled")
            ),
        }
        for run in RUNS
    ]
).set_index("Method")

results.style.format("{:.4f}", na_rep="pending")